# Temporal Fusion Transformer (TFT)

Part of the reference implementation for *Reliability-Aware Model-Dependent Uncertainty Placement under Data Asymmetry for Probabilistic Load Forecasting*. Both uncertainty placements are produced: the modular placement (post-hoc contextual residual quantiles, route 1) and the integrated placement (in-model quantiles, route 2).

Run the cells in order. Outputs go to `outputs/`. A GPU is recommended.

## 0. Environment

In [ ]:
#!pip install darts
!pip install pytorch_lightning

## 1. Symmetric protocol ($X = X'$)
Training and testing at 5-minute resolution with all channels. Writes `route1` (point + modular quantiles) and `route2` (integrated quantiles) CSVs for Dec 24 and Dec 30, 2024.

In [ ]:
# ---- repository paths -------------------------------------------------
# REPO defaults to the parent of notebooks/ when run locally, or to
# MyDrive/uncertainty-placement on Colab. Override with SEGAN_REPO.
import os
try:
    from google.colab import drive
    drive.mount("/content/drive")
    _default_repo = "/content/drive/MyDrive/uncertainty-placement"
except ImportError:
    _default_repo = os.path.abspath(os.path.join(os.getcwd(), ".."))
REPO     = os.environ.get("SEGAN_REPO", _default_repo)
DATA_DIR = os.environ.get("SEGAN_DATA", os.path.join(REPO, "data"))
OUT_ROOT = os.environ.get("SEGAN_OUT",  os.path.join(REPO, "outputs"))
os.makedirs(OUT_ROOT, exist_ok=True)
# -------------------------------------------------------------------------
# ============================
# TFT symmetric evaluation at 5-minute multivariate resolution
# FEATURE-ALIGNED + STANDARDIZED ROUTE1 + RECURSIVE TEST FORECAST VERSION
#
# Train: V1 only
# Validate: V2 excluding Dec 24 and Dec 30
# Test: Dec 24 and Dec 30 from V2
#
# Route1:
#   - deterministic TFT trained on V1 only
#   - forward one-step validation rollout
#   - residual buckets from validation residuals
#
# Route2:
#   - probabilistic TFT with recursive one-step rollout
#
# Forecasting mode alignment:
#   - recursive one-step rollout for BOTH Route1 and Route2
# ============================


# !pip -q install -U darts pytorch-lightning torch pandas numpy scikit-learn

# -------------------------------------------------
# 0) Environment check
# -------------------------------------------------
import numpy as np
import pandas as pd
import torch
import darts

print("Environment OK")
print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("torch :", torch.__version__)
print("darts :", darts.__version__)

# -------------------------------------------------
# 1) Imports
# -------------------------------------------------
import os
import time
import random
import warnings

from darts import TimeSeries
from darts.models import TFTModel
from darts.dataprocessing.transformers import Scaler
from darts.utils.likelihood_models import QuantileRegression
from pytorch_lightning.callbacks import EarlyStopping

warnings.filterwarnings("ignore")
import logging
logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)
logging.getLogger("lightning_fabric").setLevel(logging.ERROR)
logging.getLogger("lightning").setLevel(logging.ERROR)

# -------------------------------------------------
# 2) Reproducibility
# -------------------------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# -------------------------------------------------
# 3) Paths
# -------------------------------------------------
V1_PATH = os.path.join(DATA_DIR, "data_v1.0.csv")
V2_PATH = os.path.join(DATA_DIR, "data_v2.0.csv")

OUT_DIR = OUT_ROOT + "/tft_out_symmetric_5min_recursive"
os.makedirs(OUT_DIR, exist_ok=True)

MERGED_ROUTE1 = os.path.join(OUT_DIR, "merged_route1_point_uq_series_tft_symmetric_5min.csv")
MERGED_ROUTE2 = os.path.join(OUT_DIR, "merged_route2_predictions_prob_tft_symmetric_5min.csv")

# -------------------------------------------------
# 4) Timestamp rules
# -------------------------------------------------
V1_START_DATE = "2023-12-01 00:00:00"
V2_START_DATE = "2024-12-01 00:00:00"
SRC_FREQ = "5min"

# -------------------------------------------------
# 5) Forecast / model settings
# -------------------------------------------------
STEPS_PER_HOUR = 12
HORIZON_HOURS = 24

PRED_LEN = HORIZON_HOURS * STEPS_PER_HOUR   # 288
INPUT_CHUNK = 288                           # 24h context
OUTPUT_CHUNK = 1                            # recursive one-step forecast

TEST_DAYS = [
    pd.Timestamp("2024-12-24"),
    pd.Timestamp("2024-12-30"),
]

Q_LO, Q_MID, Q_HI = 0.10, 0.50, 0.90
SAMPLES = 80

EPOCHS = 8
DET_HIDDEN = 8
DET_LAYERS = 1
PROB_HIDDEN = 8
PROB_LAYERS = 1

TARGET = "power_w"

FEATURES = [
    "temp_c",
    "voltage_v",
    "amperage_a",
    "generation_w",
    "hour",
    "dayofweek",
    "minute_of_day",
    "slot_5min",
    "lag_5min",
    "lag_10min",
    "lag_1h",
    "lag_24h",
    "ma_1h",
    "ma_3h",
]

# -------------------------------------------------
# 6) Helpers
# -------------------------------------------------
def _find_col(df, candidates):
    cols = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in cols:
            return cols[cand.lower()]
    for c in df.columns:
        cl = c.lower()
        for cand in candidates:
            if cand.lower() in cl:
                return c
    return None

def enforce_non_crossing(q10, q50, q90):
    qs = np.sort(np.vstack([q10, q50, q90]).T, axis=1)
    return qs[:, 0], qs[:, 1], qs[:, 2]

def merge_daily_route1(day_files, out_path):
    parts = []
    for p in day_files:
        df = pd.read_csv(p)
        df.columns = [c.strip() for c in df.columns]
        parts.append(df)
    merged = pd.concat(parts, ignore_index=True)
    merged["timestamp"] = pd.to_datetime(merged["timestamp"], errors="coerce")
    merged = merged.sort_values("timestamp").reset_index(drop=True)
    merged.to_csv(out_path, index=False)
    return merged

def merge_daily_route2(day_files, out_path):
    parts = []
    for p in day_files:
        df = pd.read_csv(p)
        df.columns = [c.strip() for c in df.columns]
        parts.append(df)
    merged = pd.concat(parts, ignore_index=True)
    merged["timestamp"] = pd.to_datetime(merged["timestamp"], errors="coerce")
    merged = merged.sort_values("timestamp").reset_index(drop=True)

    out = pd.DataFrame({
        "timestamp": merged["timestamp"],
        "Actual Power (W)": merged["actual"],
        "q10 (W)": merged["q10"],
        "q50 (W)": merged["q50"],
        "q90 (W)": merged["q90"],
    })
    out.to_csv(out_path, index=False)
    return out

def load_raw_multivar_5min(path, start_date, src_freq="5min"):
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]

    p_col = _find_col(df, ["power (W)", "power", "load", "active power"])
    t_col = _find_col(df, ["temperature (ºC)", "temperature (C)", "temperature", "temp", "air temperature"])
    v_col = _find_col(df, ["voltage (V)", "voltage"])
    a_col = _find_col(df, ["amperage (A)", "amperage", "current"])
    g_col = _find_col(df, ["generation (w)", "generation", "pv generation"])

    required = {
        "power_w": p_col,
        "temp_c": t_col,
        "voltage_v": v_col,
        "amperage_a": a_col,
        "generation_w": g_col,
    }

    missing = [k for k, v in required.items() if v is None]
    if missing:
        raise ValueError(f"Missing required columns in {path}: {missing}. Found {df.columns.tolist()}")

    print(
        f"[{os.path.basename(path)}] "
        f"power='{p_col}', temp='{t_col}', voltage='{v_col}', amperage='{a_col}', generation='{g_col}'"
    )

    df["timestamp"] = pd.date_range(start=start_date, periods=len(df), freq=src_freq)

    for src in [p_col, t_col, v_col, a_col, g_col]:
        df[src] = pd.to_numeric(df[src], errors="coerce")

    out = df.rename(columns={
        p_col: "power_w",
        t_col: "temp_c",
        v_col: "voltage_v",
        a_col: "amperage_a",
        g_col: "generation_w",
    })[["timestamp", "power_w", "temp_c", "voltage_v", "amperage_a", "generation_w"]].copy()

    out["hour"] = out["timestamp"].dt.hour
    out["dayofweek"] = out["timestamp"].dt.dayofweek
    out["minute_of_day"] = out["timestamp"].dt.hour * 60 + out["timestamp"].dt.minute
    out["slot_5min"] = out["minute_of_day"] // 5

    return out

def force_5min_grid(df):
    df = df.copy()
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df = df.sort_values("timestamp").drop_duplicates("timestamp").reset_index(drop=True)

    full_idx = pd.date_range(
        start=df["timestamp"].min(),
        end=df["timestamp"].max(),
        freq="5min"
    )

    df = (
        df.set_index("timestamp")
          .reindex(full_idx)
          .rename_axis("timestamp")
          .reset_index()
    )

    num_cols = [c for c in df.columns if c != "timestamp"]
    for c in num_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    df[num_cols] = df[num_cols].interpolate(method="linear", limit_direction="both")
    df[num_cols] = df[num_cols].bfill().ffill()

    df["hour"] = df["timestamp"].dt.hour
    df["dayofweek"] = df["timestamp"].dt.dayofweek
    df["minute_of_day"] = df["timestamp"].dt.hour * 60 + df["timestamp"].dt.minute
    df["slot_5min"] = df["minute_of_day"] // 5

    return df

def add_engineered_features_5min(df):
    df = df.copy().sort_values("timestamp").reset_index(drop=True)

    df["lag_5min"] = df["power_w"].shift(1)
    df["lag_10min"] = df["power_w"].shift(2)
    df["lag_1h"] = df["power_w"].shift(12)
    df["lag_24h"] = df["power_w"].shift(288)

    df["ma_1h"] = df["power_w"].shift(1).rolling(window=12, min_periods=1).mean()
    df["ma_3h"] = df["power_w"].shift(1).rolling(window=36, min_periods=1).mean()

    df = df.bfill().ffill()
    return df

def to_series_and_scale(train_df, val_df, test_df):
    train_df = force_5min_grid(train_df)
    val_df   = force_5min_grid(val_df)
    test_df  = force_5min_grid(test_df)

    ts_train_y = TimeSeries.from_dataframe(
        train_df, time_col="timestamp", value_cols=[TARGET],
        fill_missing_dates=True, freq="5min"
    )
    ts_val_y = TimeSeries.from_dataframe(
        val_df, time_col="timestamp", value_cols=[TARGET],
        fill_missing_dates=True, freq="5min"
    )
    ts_test_y = TimeSeries.from_dataframe(
        test_df, time_col="timestamp", value_cols=[TARGET],
        fill_missing_dates=True, freq="5min"
    )

    ts_train_cov = TimeSeries.from_dataframe(
        train_df, time_col="timestamp", value_cols=FEATURES,
        fill_missing_dates=True, freq="5min"
    )
    ts_val_cov = TimeSeries.from_dataframe(
        val_df, time_col="timestamp", value_cols=FEATURES,
        fill_missing_dates=True, freq="5min"
    )
    ts_test_cov = TimeSeries.from_dataframe(
        test_df, time_col="timestamp", value_cols=FEATURES,
        fill_missing_dates=True, freq="5min"
    )

    scaler_y = Scaler()
    scaler_cov = Scaler()

    ts_train_y_s = scaler_y.fit_transform(ts_train_y)
    ts_val_y_s   = scaler_y.transform(ts_val_y)
    ts_test_y_s  = scaler_y.transform(ts_test_y)

    ts_train_cov_s = scaler_cov.fit_transform(ts_train_cov)
    ts_val_cov_s   = scaler_cov.transform(ts_val_cov)
    ts_test_cov_s  = scaler_cov.transform(ts_test_cov)

    return {
        "ts_train_y": ts_train_y, "ts_val_y": ts_val_y, "ts_test_y": ts_test_y,
        "ts_train_cov": ts_train_cov, "ts_val_cov": ts_val_cov, "ts_test_cov": ts_test_cov,
        "ts_train_y_s": ts_train_y_s, "ts_val_y_s": ts_val_y_s, "ts_test_y_s": ts_test_y_s,
        "ts_train_cov_s": ts_train_cov_s, "ts_val_cov_s": ts_val_cov_s, "ts_test_cov_s": ts_test_cov_s,
        "scaler_y": scaler_y, "scaler_cov": scaler_cov
    }

def build_det_tft():
    early_stopper = EarlyStopping(
        monitor="val_loss",
        patience=3,
        min_delta=0.001,
        mode="min"
    )
    trainer_args = {
        "callbacks": [early_stopper],
        "enable_checkpointing": False,
        "logger": False
    }

    model = TFTModel(
        input_chunk_length=INPUT_CHUNK,
        output_chunk_length=OUTPUT_CHUNK,
        hidden_size=DET_HIDDEN,
        lstm_layers=DET_LAYERS,
        num_attention_heads=2,
        dropout=0.2,
        batch_size=16,
        n_epochs=EPOCHS,
        add_relative_index=True,
        random_state=SEED,
        optimizer_kwargs={"lr": 3e-4},
        pl_trainer_kwargs=trainer_args,
        save_checkpoints=False,
        force_reset=True
    )
    return model

def build_prob_tft():
    early_stopper = EarlyStopping(
        monitor="val_loss",
        patience=3,
        min_delta=0.001,
        mode="min"
    )
    trainer_args = {
        "callbacks": [early_stopper],
        "enable_checkpointing": False,
        "logger": False
    }

    model = TFTModel(
        input_chunk_length=INPUT_CHUNK,
        output_chunk_length=OUTPUT_CHUNK,
        hidden_size=PROB_HIDDEN,
        lstm_layers=PROB_LAYERS,
        num_attention_heads=2,
        dropout=0.2,
        batch_size=16,
        n_epochs=EPOCHS,
        add_relative_index=True,
        random_state=SEED,
        optimizer_kwargs={"lr": 3e-4},
        pl_trainer_kwargs=trainer_args,
        likelihood=QuantileRegression(quantiles=[0.1, 0.5, 0.9]),
        save_checkpoints=False,
        force_reset=True
    )
    return model

def fit_det_tft(model, S):
    model.fit(
        series=S["ts_train_y_s"],
        val_series=S["ts_val_y_s"],
        past_covariates=S["ts_train_cov_s"],
        future_covariates=S["ts_train_cov_s"],
        val_past_covariates=S["ts_val_cov_s"],
        val_future_covariates=S["ts_val_cov_s"],
        verbose=False
    )

def fit_prob_tft(model, S):
    model.fit(
        series=S["ts_train_y_s"],
        val_series=S["ts_val_y_s"],
        past_covariates=S["ts_train_cov_s"],
        future_covariates=S["ts_train_cov_s"],
        val_past_covariates=S["ts_val_cov_s"],
        val_future_covariates=S["ts_val_cov_s"],
        verbose=False
    )

def make_recursive_predict_inputs(train_hist_df, scaler_y, scaler_cov):
    train_hist_df = force_5min_grid(train_hist_df)

    hist_tail = train_hist_df.tail(INPUT_CHUNK).copy()
    if len(hist_tail) < INPUT_CHUNK:
        raise ValueError(f"Need at least {INPUT_CHUNK} history rows, got {len(hist_tail)}")

    ts_hist_tail_y = TimeSeries.from_dataframe(
        hist_tail, time_col="timestamp", value_cols=[TARGET],
        fill_missing_dates=True, freq="5min"
    )
    ts_hist_tail_cov = TimeSeries.from_dataframe(
        hist_tail, time_col="timestamp", value_cols=FEATURES,
        fill_missing_dates=True, freq="5min"
    )

    ts_hist_tail_y_s = scaler_y.transform(ts_hist_tail_y)
    ts_hist_tail_cov_s = scaler_cov.transform(ts_hist_tail_cov)

    return ts_hist_tail_y_s, ts_hist_tail_cov_s

def one_step_det_predict(model_det, hist_y_s, hist_cov_s, next_cov_s, scaler_y):
    cov_combined = hist_cov_s.concatenate(next_cov_s)
    fc_s = model_det.predict(
        n=1,
        series=hist_y_s,
        past_covariates=cov_combined,
        future_covariates=cov_combined
    )
    fc = scaler_y.inverse_transform(fc_s)
    point = float(fc.values().flatten()[0])
    return point

def one_step_prob_predict(prob_model, hist_y_s, hist_cov_s, next_cov_s, scaler_y, num_samples=100):
    cov_combined = hist_cov_s.concatenate(next_cov_s)
    fc_samples_s = prob_model.predict(
        n=1,
        series=hist_y_s,
        past_covariates=cov_combined,
        future_covariates=cov_combined,
        num_samples=num_samples
    )
    fc_samples = scaler_y.inverse_transform(fc_samples_s)
    vals = fc_samples.all_values(copy=False)   # [time, component, sample]
    samples = vals[:, 0, :]                    # [1, sample]

    q10 = float(np.quantile(samples[0], 0.10))
    q50 = float(np.quantile(samples[0], 0.50))
    q90 = float(np.quantile(samples[0], 0.90))
    aa, bb, cc = enforce_non_crossing(np.array([q10]), np.array([q50]), np.array([q90]))
    return float(aa[0]), float(bb[0]), float(cc[0])

def recursive_forward_validation_predictions_det(
    model_det,
    full_hist_before_val,
    val_df_5m,
    scaler_y,
    scaler_cov,
    stride=1
):
    """
    Standardized Route1:
    - forward one-step validation rollout
    - update with ACTUAL validation observation
    """
    val_df_5m = force_5min_grid(val_df_5m.copy().sort_values("timestamp").reset_index(drop=True))
    hist_df = force_5min_grid(full_hist_before_val.copy().sort_values("timestamp").reset_index(drop=True))

    preds, actuals, ts_list = [], [], []

    for j in range(len(val_df_5m)):
        if j % 500 == 0:
            print(f"  validation rolling progress: {j}/{len(val_df_5m)}")

        hist_y_s, hist_cov_s = make_recursive_predict_inputs(
            train_hist_df=hist_df,
            scaler_y=scaler_y,
            scaler_cov=scaler_cov
        )

        next_row = val_df_5m.iloc[[j]].copy()
        next_cov = TimeSeries.from_dataframe(
            next_row, time_col="timestamp", value_cols=FEATURES,
            fill_missing_dates=True, freq="5min"
        )
        next_cov_s = scaler_cov.transform(next_cov)

        pred = one_step_det_predict(
            model_det=model_det,
            hist_y_s=hist_y_s,
            hist_cov_s=hist_cov_s,
            next_cov_s=next_cov_s,
            scaler_y=scaler_y
        )

        y_true = float(next_row[TARGET].iloc[0])
        ts = pd.Timestamp(next_row["timestamp"].iloc[0])

        if (j % stride) == 0:
            preds.append(pred)
            actuals.append(y_true)
            ts_list.append(ts)

        # update with ACTUAL observation
        hist_df = pd.concat([hist_df, next_row], ignore_index=True)
        hist_df = hist_df.sort_values("timestamp").drop_duplicates("timestamp").reset_index(drop=True)

    out = pd.DataFrame({
        "timestamp": ts_list,
        "actual": actuals,
        "pred": preds
    })
    return out

def build_route1_residual_buckets_from_val_recursive(
    model_det,
    full_hist_before_val,
    val_df_5m,
    scaler_y,
    scaler_cov
):
    df_res = recursive_forward_validation_predictions_det(
        model_det=model_det,
        full_hist_before_val=full_hist_before_val,
        val_df_5m=val_df_5m,
        scaler_y=scaler_y,
        scaler_cov=scaler_cov,
        stride=1
    )

    df_res["slot_5min"] = df_res["timestamp"].dt.hour * 12 + (df_res["timestamp"].dt.minute // 5)
    df_res["dayofweek"] = df_res["timestamp"].dt.dayofweek
    df_res["residual"] = df_res["actual"] - df_res["pred"]

    clip_val = np.percentile(np.abs(df_res["residual"].to_numpy()), 99)
    df_res["residual"] = np.clip(df_res["residual"], -clip_val, clip_val)

    res_q = (
        df_res.groupby(["slot_5min", "dayofweek"])["residual"]
              .quantile([Q_LO, Q_MID, Q_HI])
              .unstack()
              .rename(columns={Q_LO: "rq10", Q_MID: "rq50", Q_HI: "rq90"})
              .reset_index()
    )

    return res_q, df_res

def apply_route1_buckets(test_df, point_pred, res_q):
    tmp = test_df[["timestamp", "slot_5min", "dayofweek"]].copy().reset_index(drop=True)
    tmp["point"] = point_pred
    tmp = tmp.merge(res_q, on=["slot_5min", "dayofweek"], how="left")
    tmp[["rq10", "rq50", "rq90"]] = tmp[["rq10", "rq50", "rq90"]].fillna(0.0)

    p10 = tmp["point"].to_numpy() + tmp["rq10"].to_numpy()
    p50 = tmp["point"].to_numpy() + tmp["rq50"].to_numpy()
    p90 = tmp["point"].to_numpy() + tmp["rq90"].to_numpy()

    return enforce_non_crossing(p10, p50, p90)

def recursive_test_forecast_det(model_det, hist_df, test_df, scaler_y, scaler_cov):
    hist_df = force_5min_grid(hist_df.copy().sort_values("timestamp").reset_index(drop=True))
    test_df = force_5min_grid(test_df.copy().sort_values("timestamp").reset_index(drop=True))

    preds = []

    for h in range(len(test_df)):
        if h % 50 == 0:
            print(f"  recursive det test progress: {h}/{len(test_df)}")

        hist_y_s, hist_cov_s = make_recursive_predict_inputs(
            train_hist_df=hist_df,
            scaler_y=scaler_y,
            scaler_cov=scaler_cov
        )

        next_row = test_df.iloc[[h]].copy()
        next_cov = TimeSeries.from_dataframe(
            next_row, time_col="timestamp", value_cols=FEATURES,
            fill_missing_dates=True, freq="5min"
        )
        next_cov_s = scaler_cov.transform(next_cov)

        pred = one_step_det_predict(
            model_det=model_det,
            hist_y_s=hist_y_s,
            hist_cov_s=hist_cov_s,
            next_cov_s=next_cov_s,
            scaler_y=scaler_y
        )

        preds.append(pred)

        # recursive update with predicted point
        next_row[TARGET] = pred
        hist_df = pd.concat([hist_df, next_row], ignore_index=True)
        hist_df = hist_df.sort_values("timestamp").drop_duplicates("timestamp").reset_index(drop=True)

    return np.asarray(preds, dtype=float)

def recursive_test_forecast_prob(prob_model, hist_df, test_df, scaler_y, scaler_cov, num_samples=100):
    hist_df = force_5min_grid(hist_df.copy().sort_values("timestamp").reset_index(drop=True))
    test_df = force_5min_grid(test_df.copy().sort_values("timestamp").reset_index(drop=True))

    q10_list, q50_list, q90_list = [], [], []

    for h in range(len(test_df)):
        if h % 50 == 0:
            print(f"  recursive prob test progress: {h}/{len(test_df)}")

        hist_y_s, hist_cov_s = make_recursive_predict_inputs(
            train_hist_df=hist_df,
            scaler_y=scaler_y,
            scaler_cov=scaler_cov
        )

        next_row = test_df.iloc[[h]].copy()
        next_cov = TimeSeries.from_dataframe(
            next_row, time_col="timestamp", value_cols=FEATURES,
            fill_missing_dates=True, freq="5min"
        )
        next_cov_s = scaler_cov.transform(next_cov)

        q10, q50, q90 = one_step_prob_predict(
            prob_model=prob_model,
            hist_y_s=hist_y_s,
            hist_cov_s=hist_cov_s,
            next_cov_s=next_cov_s,
            scaler_y=scaler_y,
            num_samples=num_samples
        )

        q10_list.append(q10)
        q50_list.append(q50)
        q90_list.append(q90)

        # recursive update with predicted median
        next_row[TARGET] = q50
        hist_df = pd.concat([hist_df, next_row], ignore_index=True)
        hist_df = hist_df.sort_values("timestamp").drop_duplicates("timestamp").reset_index(drop=True)

    return (
        np.asarray(q10_list, dtype=float),
        np.asarray(q50_list, dtype=float),
        np.asarray(q90_list, dtype=float)
    )

# -------------------------------------------------
# 7) Load and preprocess V1, V2 at 5-minute resolution
# -------------------------------------------------
print("Loading raw V1 and V2 at 5-minute resolution ...")
v1_5m = load_raw_multivar_5min(V1_PATH, V1_START_DATE, SRC_FREQ)
v2_5m = load_raw_multivar_5min(V2_PATH, V2_START_DATE, SRC_FREQ)

full_5m = pd.concat([v1_5m, v2_5m], ignore_index=True)
full_5m = full_5m.sort_values("timestamp").drop_duplicates("timestamp").reset_index(drop=True)
full_5m = force_5min_grid(full_5m)
full_5m = add_engineered_features_5min(full_5m)

v1_5m = full_5m[full_5m["timestamp"] < pd.Timestamp(V2_START_DATE)].copy().reset_index(drop=True)
v2_5m = full_5m[full_5m["timestamp"] >= pd.Timestamp(V2_START_DATE)].copy().reset_index(drop=True)

print("✅ V1 5-min:", len(v1_5m))
print("✅ V2 5-min:", len(v2_5m))
print("✅ Full 5-min:", len(full_5m))

# -------------------------------------------------
# 8) Symmetric split
# -------------------------------------------------
test_mask = v2_5m["timestamp"].dt.normalize().isin(TEST_DAYS)

train_df = v1_5m.copy().reset_index(drop=True)
val_df = v2_5m[~test_mask].copy().reset_index(drop=True)
test_df_all = v2_5m[test_mask].copy().reset_index(drop=True)

print("\nSplit sizes:")
print("Train rows:", len(train_df))
print("Val rows  :", len(val_df))
print("Test rows :", len(test_df_all))

for d in TEST_DAYS:
    cur = test_df_all[test_df_all["timestamp"].dt.normalize() == d].copy()
    if len(cur) != PRED_LEN:
        raise ValueError(f"Test day {d.date()} has {len(cur)} rows, expected {PRED_LEN}.")

# -------------------------------------------------
# 9) Build TFT train/val on V1 only
# -------------------------------------------------
split = int(0.90 * len(train_df))
tr_df = train_df.iloc[:split].copy()
va_df = train_df.iloc[split:].copy()

if len(va_df) < max(INPUT_CHUNK + OUTPUT_CHUNK, 576):
    va_df = train_df.iloc[-max(INPUT_CHUNK + OUTPUT_CHUNK, 1152):].copy()
    tr_df = train_df.iloc[:-len(va_df)].copy()

dummy_test_df = val_df.iloc[:max(PRED_LEN, 1)].copy().reset_index(drop=True)

print("\nTFT train/val sizes:")
print("Train rows:", len(tr_df))
print("Val rows  :", len(va_df))
print("Dummy test:", len(dummy_test_df))

S = to_series_and_scale(tr_df, va_df, dummy_test_df)

# -------------------------------------------------
# 10) Train deterministic TFT once on V1 only
# -------------------------------------------------
from darts.utils.likelihood_models import QuantileRegression
import torch.serialization

torch.serialization.add_safe_globals([QuantileRegression])

det_model_path = os.path.join(OUT_DIR, "tft_det_symmetric_5min_recursive.pth")

if os.path.exists(det_model_path):
    print("\nLoading existing deterministic TFT model ...")
    from darts.utils.likelihood_models import QuantileRegression
    import torch.serialization
    torch.serialization.add_safe_globals([QuantileRegression])

    det_model = TFTModel.load(det_model_path, weights_only=False)
else:
    print("\nTraining deterministic TFT on V1 only ...")
    det_model = build_det_tft()

    t0 = time.time()
    fit_det_tft(det_model, S)
    det_fit_sec = time.time() - t0
    print(f"Deterministic TFT fit time: {det_fit_sec:.2f}s")

    det_model.save(det_model_path)
    print("Saved deterministic model:", det_model_path)

# -------------------------------------------------
# 11) Build Route1 residual buckets from normal V2 validation
# -------------------------------------------------
print("\nBuilding Route1 residual buckets from normal V2 validation using recursive one-step rollout ...")
full_hist_before_val = full_5m[full_5m["timestamp"] < val_df["timestamp"].min()].copy().reset_index(drop=True)

res_q, df_res = build_route1_residual_buckets_from_val_recursive(
    model_det=det_model,
    full_hist_before_val=full_hist_before_val,
    val_df_5m=val_df,
    scaler_y=S["scaler_y"],
    scaler_cov=S["scaler_cov"]
)

resq_path = os.path.join(OUT_DIR, "tft_route1_residual_quantiles_from_v2val_5min_recursive.csv")
residuals_path = os.path.join(OUT_DIR, "tft_route1_residuals_from_v2val_5min_recursive.csv")
res_q.to_csv(resq_path, index=False)
df_res.to_csv(residuals_path, index=False)

print("Saved residual quantiles:", resq_path)
print("Saved residual table    :", residuals_path)

# -------------------------------------------------
# 12) Train probabilistic TFT once on V1 only
# -------------------------------------------------
print("\nTraining probabilistic TFT on V1 only ...")
prob_model = build_prob_tft()

t0 = time.time()
fit_prob_tft(prob_model, S)
prob_fit_sec = time.time() - t0
print(f"Probabilistic TFT fit time: {prob_fit_sec:.2f}s")

prob_model_path = os.path.join(OUT_DIR, "tft_prob_symmetric_5min_recursive.pth")
prob_model.save(prob_model_path)
print("Saved probabilistic model:", prob_model_path)

# -------------------------------------------------
# 13) Test on Dec 24 and Dec 30
# -------------------------------------------------
print("\nTesting on Dec 24 and Dec 30 with recursive one-step rollout ...")

r1_uq_day_files = []
r2_prob_day_files = []

for k, day_start in enumerate(TEST_DAYS, start=1):
    print("\n" + "=" * 80)
    print(f"Test day {k}: {day_start.date()}")

    day_end = day_start + pd.Timedelta(days=1)

    test_df = test_df_all[
        (test_df_all["timestamp"] >= day_start) &
        (test_df_all["timestamp"] < day_end)
    ].copy().reset_index(drop=True)

    if len(test_df) != PRED_LEN:
        raise ValueError(f"Test day {day_start.date()} does not have {PRED_LEN} rows.")

    hist_df = full_5m[full_5m["timestamp"] <= (day_start - pd.Timedelta(minutes=5))].copy().reset_index(drop=True)

    # Route1 point recursive forecast
    t_pred_det = time.time()
    point_pred = recursive_test_forecast_det(
        model_det=det_model,
        hist_df=hist_df,
        test_df=test_df,
        scaler_y=S["scaler_y"],
        scaler_cov=S["scaler_cov"]
    )
    pred_det_sec = time.time() - t_pred_det

    y_true = test_df[TARGET].to_numpy(dtype=float)
    p10, p50, p90 = apply_route1_buckets(test_df, point_pred, res_q)

    out_r1 = pd.DataFrame({
        "timestamp": test_df["timestamp"].values,
        "actual": y_true,
        "point_pred": point_pred,
        "q10": p10,
        "q50": p50,
        "q90": p90
    })
    p_r1 = os.path.join(OUT_DIR, f"tft_route1_uq_symmetric_5min_day{k}.csv")
    out_r1.to_csv(p_r1, index=False)
    r1_uq_day_files.append(p_r1)

    # Route2 probabilistic recursive forecast
    t_pred_prob = time.time()
    q10, q50, q90 = recursive_test_forecast_prob(
        prob_model=prob_model,
        hist_df=hist_df,
        test_df=test_df,
        scaler_y=S["scaler_y"],
        scaler_cov=S["scaler_cov"],
        num_samples=SAMPLES
    )
    pred_prob_sec = time.time() - t_pred_prob

    out_r2 = pd.DataFrame({
        "timestamp": test_df["timestamp"].values,
        "actual": y_true,
        "q10": q10,
        "q50": q50,
        "q90": q90
    })
    p_r2 = os.path.join(OUT_DIR, f"tft_route2_prob_symmetric_5min_day{k}.csv")
    out_r2.to_csv(p_r2, index=False)
    r2_prob_day_files.append(p_r2)

    print("Saved:")
    print("  Route1_UQ  :", p_r1)
    print("  Route2_Prob:", p_r2)
    print(f"  Prediction times: det={pred_det_sec:.2f}s, prob={pred_prob_sec:.2f}s")

# -------------------------------------------------
# 14) Merge outputs
# -------------------------------------------------
print("\nMerging outputs ...")

merged_route1_df = merge_daily_route1(r1_uq_day_files, MERGED_ROUTE1)
merged_route2_df = merge_daily_route2(r2_prob_day_files, MERGED_ROUTE2)

print("✅ Merged files written:")
print("  Route1:", MERGED_ROUTE1)
print("  Route2:", MERGED_ROUTE2)
print("  OUT_DIR:", OUT_DIR)

print("\nRoute1 head:")
print(merged_route1_df.head())

print("\nRoute2 head:")
print(merged_route2_df.head())

## 2. Asymmetric protocol ($X \neq X'$)
Training on the full multivariate history; deployment on hourly data with temperature and calendar features only. Voltage, current and PV generation are filled with hour-of-day means from $D_1$.

In [ ]:
# ---- repository paths -------------------------------------------------
# REPO defaults to the parent of notebooks/ when run locally, or to
# MyDrive/uncertainty-placement on Colab. Override with SEGAN_REPO.
import os
try:
    from google.colab import drive
    drive.mount("/content/drive")
    _default_repo = "/content/drive/MyDrive/uncertainty-placement"
except ImportError:
    _default_repo = os.path.abspath(os.path.join(os.getcwd(), ".."))
REPO     = os.environ.get("SEGAN_REPO", _default_repo)
DATA_DIR = os.environ.get("SEGAN_DATA", os.path.join(REPO, "data"))
OUT_ROOT = os.environ.get("SEGAN_OUT",  os.path.join(REPO, "outputs"))
os.makedirs(OUT_ROOT, exist_ok=True)
# -------------------------------------------------------------------------
# ============================
# FIXED TFT ASYMMETRIC EVALUATION
#
# Goal:
#   Make TFT follow the same asymmetric evaluation logic as the
#   XGBoost / BiLSTM / BiGRU-LSTM family:
#
#   - Train source: V1 only
#   - Validation source: V2 excluding Dec 24 and Dec 30
#   - Test source: Dec 24 and Dec 30 from V2
#   - Deployment regime: HOURLY
#   - Only temperature + time are assumed available at deployment
#   - Missing variables are filled using V1-derived priors
#
# Route1:
#   deterministic TFT + post-hoc residual buckets from degraded V2 validation
#
# Route2:
#   probabilistic TFT with q10/q50/q90 under the same degraded hourly deployment
#
# Notes:
#   - This version is HOURLY all the way for TFT evaluation.
#   - That removes the old inconsistency where hourly deployment was expanded
#     back to 5-minute before prediction.
# ============================


# !pip -q install -U darts pytorch-lightning torch pandas numpy scikit-learn joblib

import os
import json
import time
import random
import warnings
import joblib
import numpy as np
import pandas as pd
import torch
import darts

from darts import TimeSeries
from darts.models import TFTModel
from darts.dataprocessing.transformers import Scaler
from darts.utils.likelihood_models import QuantileRegression
from pytorch_lightning.callbacks import EarlyStopping

warnings.filterwarnings("ignore")

# -------------------------------------------------
# 0) Environment
# -------------------------------------------------
print("Environment OK")
print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("torch :", torch.__version__)
print("darts :", darts.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

torch.set_float32_matmul_precision("medium")

# -------------------------------------------------
# 1) Reproducibility
# -------------------------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# -------------------------------------------------
# 2) Paths
# -------------------------------------------------
V1_PATH = os.path.join(DATA_DIR, "data_v1.0.csv")
V2_PATH = os.path.join(DATA_DIR, "data_v2.0.csv")

OUT_DIR = OUT_ROOT + "/tft_out_asymmetric_FIXED_hourly"
os.makedirs(OUT_DIR, exist_ok=True)

DET_MODEL_PATH = os.path.join(OUT_DIR, "tft_det_asymmetric_fixed_hourly.pth")
PROB_MODEL_PATH = os.path.join(OUT_DIR, "tft_prob_asymmetric_fixed_hourly.pth")

SCALER_Y_PATH = os.path.join(OUT_DIR, "scaler_y_hourly.pkl")
SCALER_COV_PATH = os.path.join(OUT_DIR, "scaler_cov_hourly.pkl")
META_PATH = os.path.join(OUT_DIR, "run_metadata.json")
VERSIONS_PATH = os.path.join(OUT_DIR, "library_versions.txt")

LOAD_SAVED_DET = True
LOAD_SAVED_PROB = False

# -------------------------------------------------
# 3) Timestamp rules
# -------------------------------------------------
V1_START_DATE = "2023-12-01 00:00:00"
V2_START_DATE = "2024-12-01 00:00:00"
SRC_FREQ = "5min"

# -------------------------------------------------
# 4) Hourly TFT settings
# -------------------------------------------------
# One week of hourly context
INPUT_CHUNK = 24
OUTPUT_CHUNK = 24

TEST_DAYS = [
    pd.Timestamp("2024-12-24"),
    pd.Timestamp("2024-12-30"),
]

EXPECTED_HOURLY_POINTS_PER_DAY = 24

Q_LO, Q_MID, Q_HI = 0.10, 0.50, 0.90
SAMPLES = 100

EPOCHS = 20
DET_HIDDEN = 16
DET_LAYERS = 1
PROB_HIDDEN = 16
PROB_LAYERS = 1

TARGET = "power_w"

# Rich HOURLY feature set for TFT
FEATURES = [
    "temp_c",
    "voltage_v",
    "amperage_a",
    "generation_w",
    "hour",
    "dayofweek",
    "lag_1h",
    "lag_2h",
    "lag_24h",
    "ma_3h",
    "ma_24h",
]

# -------------------------------------------------
# 5) Save run metadata
# -------------------------------------------------
meta = {
    "input_chunk_length": INPUT_CHUNK,
    "output_chunk_length": OUTPUT_CHUNK,
    "features": FEATURES,
    "target": TARGET,
    "epochs": EPOCHS,
    "det_hidden": DET_HIDDEN,
    "det_layers": DET_LAYERS,
    "prob_hidden": PROB_HIDDEN,
    "prob_layers": PROB_LAYERS,
    "quantiles": [0.1, 0.5, 0.9],
    "v1_path": V1_PATH,
    "v2_path": V2_PATH,
    "src_freq": SRC_FREQ,
    "test_days": [str(x.date()) for x in TEST_DAYS],
    "mode": "hourly_deployment_direct"
}
with open(META_PATH, "w") as f:
    json.dump(meta, f, indent=2)

with open(VERSIONS_PATH, "w") as f:
    f.write(f"numpy={np.__version__}\n")
    f.write(f"pandas={pd.__version__}\n")
    f.write(f"torch={torch.__version__}\n")
    f.write(f"darts={darts.__version__}\n")
    f.write(f"cuda_available={torch.cuda.is_available()}\n")
    if torch.cuda.is_available():
        f.write(f"gpu_name={torch.cuda.get_device_name(0)}\n")

print("Saved metadata:", META_PATH)
print("Saved versions:", VERSIONS_PATH)

# -------------------------------------------------
# 6) Helpers
# -------------------------------------------------
def cast_numeric_float32(df):
    df = df.copy()
    num_cols = [c for c in df.columns if c != "timestamp"]
    for c in num_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df[num_cols] = df[num_cols].astype("float32")
    return df

def _find_col(df, candidates):
    cols = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in cols:
            return cols[cand.lower()]
    for c in df.columns:
        cl = c.lower()
        for cand in candidates:
            if cand.lower() in cl:
                return c
    return None

def enforce_non_crossing(q10, q50, q90):
    qs = np.sort(np.vstack([q10, q50, q90]).T, axis=1)
    return qs[:, 0], qs[:, 1], qs[:, 2]

def merge_daily_route1(day_files, out_path):
    parts = []
    for p in day_files:
        df = pd.read_csv(p)
        df.columns = [c.strip() for c in df.columns]
        parts.append(df)
    merged = pd.concat(parts, ignore_index=True)
    merged["timestamp"] = pd.to_datetime(merged["timestamp"], errors="coerce")
    merged = merged.sort_values("timestamp").reset_index(drop=True)
    merged.to_csv(out_path, index=False)
    return merged

def merge_daily_route2(day_files, out_path):
    parts = []
    for p in day_files:
        df = pd.read_csv(p)
        df.columns = [c.strip() for c in df.columns]
        parts.append(df)
    merged = pd.concat(parts, ignore_index=True)
    merged["timestamp"] = pd.to_datetime(merged["timestamp"], errors="coerce")
    merged = merged.sort_values("timestamp").reset_index(drop=True)

    out = pd.DataFrame({
        "timestamp": merged["timestamp"],
        "Actual Power (W)": merged["actual"],
        "q10 (W)": merged["q10"],
        "q50 (W)": merged["q50"],
        "q90 (W)": merged["q90"],
    })
    out.to_csv(out_path, index=False)
    return out

def load_raw_multivar_5min(path, start_date, src_freq="5min"):
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]

    p_col = _find_col(df, ["power (W)", "power", "load", "active power"])
    t_col = _find_col(df, ["temperature (ºC)", "temperature (C)", "temperature", "temp", "air temperature"])
    v_col = _find_col(df, ["voltage (V)", "voltage"])
    a_col = _find_col(df, ["amperage (A)", "amperage", "current"])
    g_col = _find_col(df, ["generation (w)", "generation", "pv generation"])

    required = {
        "power_w": p_col,
        "temp_c": t_col,
        "voltage_v": v_col,
        "amperage_a": a_col,
        "generation_w": g_col,
    }

    missing = [k for k, v in required.items() if v is None]
    if missing:
        raise ValueError(f"Missing required columns in {path}: {missing}. Found {df.columns.tolist()}")

    print(
        f"[{os.path.basename(path)}] "
        f"power='{p_col}', temp='{t_col}', voltage='{v_col}', amperage='{a_col}', generation='{g_col}'"
    )

    df["timestamp"] = pd.date_range(start=start_date, periods=len(df), freq=src_freq)

    for src in [p_col, t_col, v_col, a_col, g_col]:
        df[src] = pd.to_numeric(df[src], errors="coerce")

    out = df.rename(columns={
        p_col: "power_w",
        t_col: "temp_c",
        v_col: "voltage_v",
        a_col: "amperage_a",
        g_col: "generation_w",
    })[["timestamp", "power_w", "temp_c", "voltage_v", "amperage_a", "generation_w"]].copy()

    out = cast_numeric_float32(out)
    return out

def force_5min_grid(df):
    df = df.copy()
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df = df.sort_values("timestamp").drop_duplicates("timestamp").reset_index(drop=True)

    full_idx = pd.date_range(
        start=df["timestamp"].min(),
        end=df["timestamp"].max(),
        freq="5min"
    )

    df = (
        df.set_index("timestamp")
          .reindex(full_idx)
          .rename_axis("timestamp")
          .reset_index()
    )

    num_cols = [c for c in df.columns if c != "timestamp"]
    for c in num_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    df[num_cols] = df[num_cols].interpolate(method="linear", limit_direction="both")
    df[num_cols] = df[num_cols].bfill().ffill()

    df = cast_numeric_float32(df)
    return df

def aggregate_to_hourly(df_5m):
    df = df_5m.copy()
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df["ts_hour"] = df["timestamp"].dt.floor("H")

    ag = (
        df.groupby("ts_hour", as_index=False)
          .agg({
              "power_w": "mean",
              "temp_c": "mean",
              "voltage_v": "mean",
              "amperage_a": "mean",
              "generation_w": "mean",
          })
          .rename(columns={"ts_hour": "timestamp"})
    )

    ag["hour"] = ag["timestamp"].dt.hour
    ag["dayofweek"] = ag["timestamp"].dt.dayofweek
    return cast_numeric_float32(ag)

def add_engineered_features_hourly(df):
    df = df.copy().sort_values("timestamp").reset_index(drop=True)

    df["hour"] = df["timestamp"].dt.hour
    df["dayofweek"] = df["timestamp"].dt.dayofweek

    df["lag_1h"] = df["power_w"].shift(1)
    df["lag_2h"] = df["power_w"].shift(2)
    df["lag_24h"] = df["power_w"].shift(24)

    df["ma_3h"] = df["power_w"].shift(1).rolling(window=3, min_periods=1).mean()
    df["ma_24h"] = df["power_w"].shift(1).rolling(window=24, min_periods=1).mean()

    df = df.bfill().ffill()
    return cast_numeric_float32(df)

def build_hourly_priors_from_train(train_h):
    priors_hour = (
        train_h.groupby("hour")[["voltage_v", "amperage_a", "generation_w"]]
               .median()
               .reset_index()
    )

    priors_global = {
        "temp_c_fallback": float(train_h["temp_c"].median()),
        "voltage_v": float(train_h["voltage_v"].median()),
        "amperage_a": float(train_h["amperage_a"].median()),
        "generation_w": float(train_h["generation_w"].median()),
        "lag_1h": float(train_h["power_w"].shift(1).median()),
        "lag_2h": float(train_h["power_w"].shift(2).median()),
        "lag_24h": float(train_h["power_w"].shift(24).median()),
        "ma_3h": float(train_h["power_w"].shift(1).rolling(3, min_periods=1).mean().median()),
        "ma_24h": float(train_h["power_w"].shift(1).rolling(24, min_periods=1).mean().median()),
    }

    return cast_numeric_float32(priors_hour), priors_global

def build_deployment_hourly_frame(hourly_df, priors_hour_df, priors_global):
    df = hourly_df.copy().reset_index(drop=True)
    pri_map = priors_hour_df.set_index("hour")[["voltage_v", "amperage_a", "generation_w"]].to_dict("index")

    rows = []
    for _, r in df.iterrows():
        ts = pd.Timestamp(r["timestamp"])
        h = int(ts.hour)
        dow = int(ts.dayofweek)

        pri = pri_map.get(h, {})
        rows.append({
            "timestamp": ts,
            "power_w": float(r["power_w"]),
            "temp_c": float(r["temp_c"]),
            "voltage_v": float(pri.get("voltage_v", priors_global["voltage_v"])),
            "amperage_a": float(pri.get("amperage_a", priors_global["amperage_a"])),
            "generation_w": float(pri.get("generation_w", priors_global["generation_w"])),
            "hour": h,
            "dayofweek": dow,
            "lag_1h": float(priors_global["lag_1h"]),
            "lag_2h": float(priors_global["lag_2h"]),
            "lag_24h": float(priors_global["lag_24h"]),
            "ma_3h": float(priors_global["ma_3h"]),
            "ma_24h": float(priors_global["ma_24h"]),
        })

    out = pd.DataFrame(rows)
    return cast_numeric_float32(out)

def to_series_and_scale(train_df, val_df, dummy_test_df):
    train_df = train_df.copy().sort_values("timestamp").reset_index(drop=True)
    val_df = val_df.copy().sort_values("timestamp").reset_index(drop=True)
    dummy_test_df = dummy_test_df.copy().sort_values("timestamp").reset_index(drop=True)

    ts_train_y = TimeSeries.from_dataframe(
        train_df, time_col="timestamp", value_cols=[TARGET],
        fill_missing_dates=True, freq="H"
    )
    ts_val_y = TimeSeries.from_dataframe(
        val_df, time_col="timestamp", value_cols=[TARGET],
        fill_missing_dates=True, freq="H"
    )
    ts_test_y = TimeSeries.from_dataframe(
        dummy_test_df, time_col="timestamp", value_cols=[TARGET],
        fill_missing_dates=True, freq="H"
    )

    ts_train_cov = TimeSeries.from_dataframe(
        train_df, time_col="timestamp", value_cols=FEATURES,
        fill_missing_dates=True, freq="H"
    )
    ts_val_cov = TimeSeries.from_dataframe(
        val_df, time_col="timestamp", value_cols=FEATURES,
        fill_missing_dates=True, freq="H"
    )
    ts_test_cov = TimeSeries.from_dataframe(
        dummy_test_df, time_col="timestamp", value_cols=FEATURES,
        fill_missing_dates=True, freq="H"
    )

    scaler_y = Scaler()
    scaler_cov = Scaler()

    ts_train_y_s = scaler_y.fit_transform(ts_train_y)
    ts_val_y_s = scaler_y.transform(ts_val_y)
    ts_test_y_s = scaler_y.transform(ts_test_y)

    ts_train_cov_s = scaler_cov.fit_transform(ts_train_cov)
    ts_val_cov_s = scaler_cov.transform(ts_val_cov)
    ts_test_cov_s = scaler_cov.transform(ts_test_cov)

    return {
        "ts_train_y": ts_train_y, "ts_val_y": ts_val_y, "ts_test_y": ts_test_y,
        "ts_train_cov": ts_train_cov, "ts_val_cov": ts_val_cov, "ts_test_cov": ts_test_cov,
        "ts_train_y_s": ts_train_y_s, "ts_val_y_s": ts_val_y_s, "ts_test_y_s": ts_test_y_s,
        "ts_train_cov_s": ts_train_cov_s, "ts_val_cov_s": ts_val_cov_s, "ts_test_cov_s": ts_test_cov_s,
        "scaler_y": scaler_y, "scaler_cov": scaler_cov
    }

def get_trainer_args():
    early_stopper = EarlyStopping(
        monitor="val_loss",
        patience=4,
        min_delta=0.001,
        mode="min"
    )

    if torch.cuda.is_available():
        trainer_args = {
            "callbacks": [early_stopper],
            "enable_checkpointing": False,
            "logger": False,
            "accelerator": "gpu",
            "devices": 1,
            "precision": 32,
        }
    else:
        trainer_args = {
            "callbacks": [early_stopper],
            "enable_checkpointing": False,
            "logger": False,
            "accelerator": "cpu",
            "devices": 1,
            "precision": 32,
        }

    return trainer_args

def build_det_tft():
    trainer_args = get_trainer_args()

    model = TFTModel(
        input_chunk_length=INPUT_CHUNK,
        output_chunk_length=OUTPUT_CHUNK,
        hidden_size=DET_HIDDEN,
        lstm_layers=DET_LAYERS,
        num_attention_heads=2,
        dropout=0.2,
        batch_size=16,
        n_epochs=EPOCHS,
        add_relative_index=True,
        random_state=SEED,
        optimizer_kwargs={"lr": 3e-4},
        pl_trainer_kwargs=trainer_args,
        save_checkpoints=False,
        force_reset=True
    )
    return model

def build_prob_tft():
    trainer_args = get_trainer_args()

    model = TFTModel(
        input_chunk_length=INPUT_CHUNK,
        output_chunk_length=OUTPUT_CHUNK,
        hidden_size=PROB_HIDDEN,
        lstm_layers=PROB_LAYERS,
        num_attention_heads=2,
        dropout=0.2,
        batch_size=16,
        n_epochs=EPOCHS,
        add_relative_index=True,
        random_state=SEED,
        optimizer_kwargs={"lr": 3e-4},
        pl_trainer_kwargs=trainer_args,
        likelihood=QuantileRegression(quantiles=[0.1, 0.5, 0.9]),
        save_checkpoints=False,
        force_reset=True
    )
    return model

def fit_det_tft(model, S):
    model.fit(
        series=S["ts_train_y_s"],
        val_series=S["ts_val_y_s"],
        past_covariates=S["ts_train_cov_s"],
        future_covariates=S["ts_train_cov_s"],
        val_past_covariates=S["ts_val_cov_s"],
        val_future_covariates=S["ts_val_cov_s"],
        verbose=True
    )

def fit_prob_tft(model, S):
    model.fit(
        series=S["ts_train_y_s"],
        val_series=S["ts_val_y_s"],
        past_covariates=S["ts_train_cov_s"],
        future_covariates=S["ts_train_cov_s"],
        val_past_covariates=S["ts_val_cov_s"],
        val_future_covariates=S["ts_val_cov_s"],
        verbose=True
    )

def make_predict_inputs_from_hist_and_deploy_hourly(hist_df_h, deploy_df_h, scaler_y, scaler_cov):
    hist_df_h = hist_df_h.copy().sort_values("timestamp").reset_index(drop=True)
    deploy_df_h = deploy_df_h.copy().sort_values("timestamp").reset_index(drop=True)

    if len(hist_df_h) < INPUT_CHUNK:
        raise ValueError(f"Need at least {INPUT_CHUNK} hourly history rows, got {len(hist_df_h)}")

    hist_tail = hist_df_h.tail(INPUT_CHUNK).copy()

    ts_hist_tail_y = TimeSeries.from_dataframe(
        hist_tail, time_col="timestamp", value_cols=[TARGET],
        fill_missing_dates=True, freq="H"
    )
    ts_hist_tail_cov = TimeSeries.from_dataframe(
        hist_tail, time_col="timestamp", value_cols=FEATURES,
        fill_missing_dates=True, freq="H"
    )
    ts_deploy_cov = TimeSeries.from_dataframe(
        deploy_df_h, time_col="timestamp", value_cols=FEATURES,
        fill_missing_dates=True, freq="H"
    )
    ts_deploy_y = TimeSeries.from_dataframe(
        deploy_df_h, time_col="timestamp", value_cols=[TARGET],
        fill_missing_dates=True, freq="H"
    )

    ts_hist_tail_y_s = scaler_y.transform(ts_hist_tail_y)
    ts_hist_tail_cov_s = scaler_cov.transform(ts_hist_tail_cov)
    ts_deploy_cov_s = scaler_cov.transform(ts_deploy_cov)
    ts_deploy_y_s = scaler_y.transform(ts_deploy_y)

    ts_cov_combined = ts_hist_tail_cov_s.concatenate(ts_deploy_cov_s)
    return ts_hist_tail_y_s, ts_cov_combined, ts_deploy_y_s

def build_route1_residual_buckets_asymmetric(
    model_det,
    train_df_h,
    val_df_h,
    priors_hour,
    priors_global,
    scaler_y,
    scaler_cov
):
    val_h_dep = build_deployment_hourly_frame(val_df_h, priors_hour, priors_global)

    hist_until_eval = train_df_h.copy().sort_values("timestamp").reset_index(drop=True)

    hist_tail_y_s, cov_combined_s, val_dep_y_s = make_predict_inputs_from_hist_and_deploy_hourly(
        hist_df_h=hist_until_eval,
        deploy_df_h=val_h_dep,
        scaler_y=scaler_y,
        scaler_cov=scaler_cov
    )

    fc_s = model_det.predict(
        n=len(val_h_dep),
        series=hist_tail_y_s,
        past_covariates=cov_combined_s,
        future_covariates=cov_combined_s
    )

    fc = scaler_y.inverse_transform(fc_s)
    actual = scaler_y.inverse_transform(val_dep_y_s)

    df_pred = pd.DataFrame({
        "timestamp": pd.to_datetime(fc.time_index),
        "pred": fc.values().flatten()
    })
    df_true = pd.DataFrame({
        "timestamp": pd.to_datetime(actual.time_index),
        "true": actual.values().flatten()
    })

    df_res = pd.merge(df_true, df_pred, on="timestamp", how="inner")
    df_res["hour"] = df_res["timestamp"].dt.hour
    df_res["dayofweek"] = df_res["timestamp"].dt.dayofweek
    df_res["residual"] = df_res["true"] - df_res["pred"]

    clip_val = np.percentile(np.abs(df_res["residual"].to_numpy()), 99)
    df_res["residual"] = np.clip(df_res["residual"], -clip_val, clip_val)

    res_q = (
        df_res.groupby(["hour", "dayofweek"])["residual"]
              .quantile([Q_LO, Q_MID, Q_HI])
              .unstack()
              .rename(columns={Q_LO: "rq10", Q_MID: "rq50", Q_HI: "rq90"})
              .reset_index()
    )

    return res_q, df_res

def apply_route1_buckets_hourly(test_timestamps, point_pred, res_q):
    tmp = pd.DataFrame({"timestamp": pd.to_datetime(test_timestamps)})
    tmp["hour"] = tmp["timestamp"].dt.hour
    tmp["dayofweek"] = tmp["timestamp"].dt.dayofweek

    tmp = tmp.merge(res_q, on=["hour", "dayofweek"], how="left")
    tmp[["rq10", "rq50", "rq90"]] = tmp[["rq10", "rq50", "rq90"]].fillna(0.0)

    p10 = point_pred + tmp["rq10"].to_numpy()
    p50 = point_pred + tmp["rq50"].to_numpy()
    p90 = point_pred + tmp["rq90"].to_numpy()

    return enforce_non_crossing(p10, p50, p90)

# -------------------------------------------------
# 7) Load raw V1/V2 at 5-minute and aggregate to hourly
# -------------------------------------------------
print("\nLoading raw V1 and V2 at 5-minute resolution ...")
v1_5m = load_raw_multivar_5min(V1_PATH, V1_START_DATE, SRC_FREQ)
v2_5m = load_raw_multivar_5min(V2_PATH, V2_START_DATE, SRC_FREQ)

full_5m = pd.concat([v1_5m, v2_5m], ignore_index=True)
full_5m = full_5m.sort_values("timestamp").drop_duplicates("timestamp").reset_index(drop=True)
full_5m = force_5min_grid(full_5m)

v1_5m = full_5m[full_5m["timestamp"] < pd.Timestamp(V2_START_DATE)].copy().reset_index(drop=True)
v2_5m = full_5m[full_5m["timestamp"] >= pd.Timestamp(V2_START_DATE)].copy().reset_index(drop=True)

v1_h = add_engineered_features_hourly(aggregate_to_hourly(v1_5m))
v2_h = add_engineered_features_hourly(aggregate_to_hourly(v2_5m))

full_h = pd.concat([v1_h, v2_h], ignore_index=True)
full_h = full_h.sort_values("timestamp").drop_duplicates("timestamp").reset_index(drop=True)

print("V1 hourly:", len(v1_h))
print("V2 hourly:", len(v2_h))
print("Full hourly:", len(full_h))

# -------------------------------------------------
# 8) Split
# -------------------------------------------------
test_mask = v2_h["timestamp"].dt.normalize().isin(TEST_DAYS)

train_df = v1_h.copy().reset_index(drop=True)
val_df = v2_h[~test_mask].copy().reset_index(drop=True)
test_df_hourly_all = v2_h[test_mask].copy().reset_index(drop=True)

print("\nSplit sizes:")
print("Train rows:", len(train_df))
print("Val rows  :", len(val_df))
print("Test rows :", len(test_df_hourly_all))

for d in TEST_DAYS:
    cur = test_df_hourly_all[test_df_hourly_all["timestamp"].dt.normalize() == d].copy()
    if len(cur) != EXPECTED_HOURLY_POINTS_PER_DAY:
        raise ValueError(f"Test day {d.date()} has {len(cur)} hourly rows, expected 24.")

# -------------------------------------------------
# 9) Build V1 priors
# -------------------------------------------------
print("\nBuilding V1 priors for deployment-time missing variables...")
priors_hour, priors_global = build_hourly_priors_from_train(train_df)

pd.DataFrame(priors_hour).to_csv(
    os.path.join(OUT_DIR, "v1_hourly_priors_for_missing_vars.csv"),
    index=False
)
pd.DataFrame([priors_global]).to_csv(
    os.path.join(OUT_DIR, "v1_global_priors_for_missing_vars.csv"),
    index=False
)

# -------------------------------------------------
# 10) TFT train/val split inside V1 only
# -------------------------------------------------
split = int(0.90 * len(train_df))
tr_df = train_df.iloc[:split].copy()
va_df = train_df.iloc[split:].copy()

min_needed = max(INPUT_CHUNK + OUTPUT_CHUNK, 24 * 14)
if len(va_df) < min_needed:
    va_df = train_df.iloc[-min_needed:].copy()
    tr_df = train_df.iloc[:-len(va_df)].copy()

dummy_test_df = val_df.iloc[:OUTPUT_CHUNK].copy().reset_index(drop=True)

print("\nTFT train/val sizes:")
print("Train rows:", len(tr_df))
print("Val rows  :", len(va_df))
print("Dummy test:", len(dummy_test_df))

S = to_series_and_scale(tr_df, va_df, dummy_test_df)

joblib.dump(S["scaler_y"], SCALER_Y_PATH)
joblib.dump(S["scaler_cov"], SCALER_COV_PATH)
print("Saved scaler_y  :", SCALER_Y_PATH)
print("Saved scaler_cov:", SCALER_COV_PATH)

# -------------------------------------------------
# 11) Deterministic TFT
# -------------------------------------------------
if LOAD_SAVED_DET and os.path.exists(DET_MODEL_PATH):
    print("\nLoading saved deterministic TFT ...")
    det_model = TFTModel.load(DET_MODEL_PATH, map_location="cpu")
    print("Loaded deterministic model:", DET_MODEL_PATH)
else:
    print("\nTraining deterministic TFT on V1 only ...")
    det_model = build_det_tft()

    t0 = time.time()
    fit_det_tft(det_model, S)
    det_fit_sec = time.time() - t0
    print(f"Deterministic TFT fit time: {det_fit_sec:.2f}s")

    det_model.save(DET_MODEL_PATH)
    print("Saved deterministic model:", DET_MODEL_PATH)

# -------------------------------------------------
# 12) Route1 residual buckets from degraded V2 validation
# -------------------------------------------------
print("\nBuilding Route1 residual buckets from V2 validation in degraded hourly mode ...")
res_q, df_res = build_route1_residual_buckets_asymmetric(
    model_det=det_model,
    train_df_h=train_df,
    val_df_h=val_df,
    priors_hour=priors_hour,
    priors_global=priors_global,
    scaler_y=S["scaler_y"],
    scaler_cov=S["scaler_cov"]
)

resq_path = os.path.join(OUT_DIR, "tft_route1_residual_quantiles_from_v2val_asymmetric_hourly.csv")
residuals_path = os.path.join(OUT_DIR, "tft_route1_residuals_from_v2val_asymmetric_hourly.csv")
res_q.to_csv(resq_path, index=False)
df_res.to_csv(residuals_path, index=False)

print("Saved residual quantiles:", resq_path)
print("Saved residual table    :", residuals_path)

# -------------------------------------------------
# 13) Probabilistic TFT
# -------------------------------------------------
if LOAD_SAVED_PROB and os.path.exists(PROB_MODEL_PATH):
    print("\nLoading saved probabilistic TFT ...")
    prob_model = TFTModel.load(PROB_MODEL_PATH, map_location="cpu")
    print("Loaded probabilistic model:", PROB_MODEL_PATH)
else:
    print("\nTraining probabilistic TFT on V1 only ...")
    prob_model = build_prob_tft()

    t0 = time.time()
    fit_prob_tft(prob_model, S)
    prob_fit_sec = time.time() - t0
    print(f"Probabilistic TFT fit time: {prob_fit_sec:.2f}s")

    prob_model.save(PROB_MODEL_PATH)
    print("Saved probabilistic model:", PROB_MODEL_PATH)

# -------------------------------------------------
# 14) Test on Dec 24 and Dec 30 in degraded hourly mode
# -------------------------------------------------
print("\nTesting on Dec 24 and Dec 30 in degraded hourly mode ...")

r1_uq_day_files = []
r2_prob_day_files = []

for k, day_start in enumerate(TEST_DAYS, start=1):
    print("\n" + "=" * 80)
    print(f"Test day {k}: {day_start.date()}")

    day_end = day_start + pd.Timedelta(days=1)

    test_h = test_df_hourly_all[
        (test_df_hourly_all["timestamp"] >= day_start) &
        (test_df_hourly_all["timestamp"] < day_end)
    ].copy().reset_index(drop=True)

    if len(test_h) != EXPECTED_HOURLY_POINTS_PER_DAY:
        raise ValueError(f"Test day {day_start.date()} does not have 24 hourly rows.")

    dep_h = build_deployment_hourly_frame(test_h, priors_hour, priors_global)

    hist_h = full_h[full_h["timestamp"] < day_start].copy().reset_index(drop=True)

    hist_tail_y_s, cov_combined_s, dep_y_s = make_predict_inputs_from_hist_and_deploy_hourly(
        hist_df_h=hist_h,
        deploy_df_h=dep_h,
        scaler_y=S["scaler_y"],
        scaler_cov=S["scaler_cov"]
    )

    # Route1 deterministic
    t_pred_det = time.time()
    det_fc_s = det_model.predict(
        n=len(dep_h),
        series=hist_tail_y_s,
        past_covariates=cov_combined_s,
        future_covariates=cov_combined_s
    )
    pred_det_sec = time.time() - t_pred_det

    det_fc = S["scaler_y"].inverse_transform(det_fc_s)
    actual = S["scaler_y"].inverse_transform(dep_y_s)

    point_pred = det_fc.values().flatten().astype(float)
    y_true = actual.values().flatten().astype(float)
    ts_hour = pd.to_datetime(det_fc.time_index)

    p10, p50, p90 = apply_route1_buckets_hourly(ts_hour, point_pred, res_q)

    out_r1 = pd.DataFrame({
        "timestamp": ts_hour,
        "actual": y_true,
        "point_pred": point_pred,
        "q10": p10,
        "q50": p50,
        "q90": p90
    })
    p_r1 = os.path.join(OUT_DIR, f"tft_route1_uq_asymmetric_hourly_day{k}.csv")
    out_r1.to_csv(p_r1, index=False)
    r1_uq_day_files.append(p_r1)

    # Route2 probabilistic
    t_pred_prob = time.time()
    fc_samples_s = prob_model.predict(
        n=len(dep_h),
        series=hist_tail_y_s,
        past_covariates=cov_combined_s,
        future_covariates=cov_combined_s,
        num_samples=SAMPLES
    )
    pred_prob_sec = time.time() - t_pred_prob

    fc_samples = S["scaler_y"].inverse_transform(fc_samples_s)
    vals = fc_samples.all_values(copy=False)   # [time, component, sample]
    samples = vals[:, 0, :]

    q10_h = np.quantile(samples, 0.10, axis=1)
    q50_h = np.quantile(samples, 0.50, axis=1)
    q90_h = np.quantile(samples, 0.90, axis=1)
    q10_h, q50_h, q90_h = enforce_non_crossing(q10_h, q50_h, q90_h)

    out_r2 = pd.DataFrame({
        "timestamp": ts_hour,
        "actual": y_true,
        "q10": q10_h,
        "q50": q50_h,
        "q90": q90_h
    })
    p_r2 = os.path.join(OUT_DIR, f"tft_route2_prob_asymmetric_hourly_day{k}.csv")
    out_r2.to_csv(p_r2, index=False)
    r2_prob_day_files.append(p_r2)

    print("Saved:")
    print("  Route1_UQ  :", p_r1)
    print("  Route2_Prob:", p_r2)
    print(f"  Prediction times: det={pred_det_sec:.2f}s, prob={pred_prob_sec:.2f}s")

# -------------------------------------------------
# 15) Merge outputs
# -------------------------------------------------
print("\nMerging outputs ...")

merged_route1_path = os.path.join(OUT_DIR, "merged_route1_point_uq_series_tft_asymmetric_hourly_FIXED.csv")
merged_route1_df = merge_daily_route1(r1_uq_day_files, merged_route1_path)

merged_route2_path = os.path.join(OUT_DIR, "merged_route2_predictions_prob_tft_asymmetric_hourly_FIXED.csv")
merged_route2_df = merge_daily_route2(r2_prob_day_files, merged_route2_path)

print("Merged files written:")
print("  Route1:", merged_route1_path)
print("  Route2:", merged_route2_path)
print("  OUT_DIR:", OUT_DIR)

print("\nRoute1 head:")
print(merged_route1_df.head())

print("\nRoute2 head:")
print(merged_route2_df.head())